In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark
from energy_calculation import evaluate_quantum_energies, evaluate_pyrosetta_energies, compare_energies
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
# Pyrosetta Relevant Code
benchmark_pose = bpti_ryfyn_benchmark()
rotamer_lib, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
generator_params = init_generator_params(h_flex_linear)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset, penalty=0.0)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
1 1
2 2
3 3
4 4
5 5
{0: 1.2480376958847046, 1: 1.5509175062179565, 2: 1.6199729442596436, 3: 1.8183133602142334} [1.2480376958847046, 1.6199729442596436, 1.5509175062179565, 1.9465504884719849, 1.8183133602142334, 2.157853841781616, 2.5786564350128174, 3.857661247253418, 4.271278381347656, 3.083498477935791, 4.202057838439941, 4.441059589385986]
{0: 2.0819952487945557, 1: 2.081996440887451, 2: 3.0249295234680176, 3: 3.024930477142334} [2.081996440887451, 2.0819952487945557, 3.024930477142334, 3.0249295234680176, 4.212233543395996, 3.185492992401123, 3.1854920387268066]
{0: 2.783350706100464, 1: 3.331935405731201, 2: 4.205162048339844} [2.783350706100464, 3.331935405731201, 4.205162048339844]
{0: 2.8021

In [3]:
def evaluate_quantum_energies(valid_conformations, h_flex, J_flex, global_offset, params):
    wire_offsets = params["wire_offsets"]

    for conformation in valid_conformations:
        bitstring = conformation["bitstring"]
        current_energy = global_offset

        # One body energies
        for seq, energies in h_flex.items():
            base_wire = wire_offsets[seq]
            for rot, e_val in enumerate(energies):
                if bitstring[base_wire + rot] == 1:
                    current_energy += e_val

        # Two body energies
        for (seq_i, seq_j), interactions in J_flex.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    current_energy += e_val

        conformation['quantum_energy'] = current_energy
    # raise NotImplementedError("Not yet implemented")

evaluate_quantum_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, params=generator_params)
valid_conformations.sort(key=lambda conf: conf['quantum_energy'])
print(valid_conformations)

[{'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(0.9999989367814385), 'quantum_energy': 4.294393211603165, 'biological_energy': None, 'pose': None}, {'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(5.716815776242786e-08), 'quantum_energy': 4.434172302484512, 'biological_energy': None, 'pose': None}, {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 'probability': np.float64(6.287148259905738e-08), 'quantum_energy': 5.069023251533508, 'biological_energy': None, 'pose': None}, {'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 'probability': np.float64(3.589882078626441e-15), 'quantum_energy': 5.208802342414856, 'biological_energy': None, 'pose': None}, {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], 'probability': np.float64(6.405762999552322e-08), 'quantum_energy': 5.2618705332279205, 'biological_energy': None, 'pose'

In [4]:
def evaluate_pyrosetta_energies(pose, valid_conformations, rotsets, scorefxn, params):
    for conformation in valid_conformations:

        bitstring = conformation["bitstring"]
        bio_energy, pose = evaluate_singular_pyrosetta_energy(pose, bitstring, rotsets, scorefxn, params)
        conformation['biological_energy'] = bio_energy
        conformation['pose'] = pose


def evaluate_singular_pyrosetta_energy(pose, bitstrings, rotsets, scorefxn, params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstrings[base_wire : base_wire + num_rots]
        local_idx = residue_bits.index(1)

        pyrosetta_rotamer_id = local_idx + 1

        rotamer_set = rotsets.rotamer_set_for_residue(seq)
        selected_residue = rotamer_set.rotamer(pyrosetta_rotamer_id)

        test_pose.replace_residue(seq, selected_residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

evaluate_pyrosetta_energies(benchmark_pose, valid_conformations, rot_sets, scorefxn, generator_params)

In [5]:
deltas = [conf['quantum_energy'] - conf['biological_energy'] for conf in valid_conformations]
valid_conformations

[{'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(0.9999989367814385),
  'quantum_energy': 4.294393211603165,
  'biological_energy': 14.099258597892437,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x7a94122a9d70>},
 {'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(5.716815776242786e-08),
  'quantum_energy': 4.434172302484512,
  'biological_energy': 13.443407878067852,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x7a94122a9ef0>},
 {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
  'probability': np.float64(6.287148259905738e-08),
  'quantum_energy': 5.069023251533508,
  'biological_energy': 14.199654180936754,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x7a94122aaaf0>},
 {'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
  'probability': np.float64(3.589882078626441e-15),
  'quantum_energy': 5.208802342414856,
  'biological_energ